In [1]:
from sklearn.datasets import load_breast_cancer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNClassifier
from tabpfn.constants import ModelVersion
from tabpfn.regressor import TabPFNRegressor

# Load data
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Initialize a classifier
# To use TabPFN v2:
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)


# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities[:, 1]))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

/home/nezvevik/projects/TabPFN/src/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


ROC AUC: 0.9984175488377168
Accuracy 0.9824561403508771


In [2]:
from sklearn.datasets import fetch_openml
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

from tabpfn import TabPFNRegressor
from tabpfn.constants import ModelVersion

# Load Boston Housing data
df = fetch_openml(data_id=531, as_frame=True)  # Boston Housing dataset
X = df.data
y = df.target.astype(float)  # Ensure target is float for regression

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)

# Initialize the regressor
# regressor = TabPFNRegressor()  # Uses TabPFN-2.5 weights, trained on synthetic data only.
# To use TabPFN v2:
regressor = TabPFNRegressor.create_default_for_version(ModelVersion.V2)
regressor.fit(X_train, y_train)

# Predict on the test set
predictions = regressor.predict(X_test)

# Evaluate the model
mse = mean_squared_error(y_test, predictions)
r2 = r2_score(y_test, predictions)

print("Mean Squared Error (MSE):", mse)
print("R² Score:", r2)

/home/nezvevik/projects/TabPFN/src/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


Mean Squared Error (MSE): 10.706196995046271
R² Score: 0.8680369519092259


In [17]:
# load the mnist dataset
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
num_data = 1000
mnist.data = mnist.data[:num_data]
mnist.target = mnist.target[:num_data]
X, y = mnist.data, mnist.target.astype(int)

# do pca to reduce dimensions
from sklearn.decomposition import PCA
pca = PCA(n_components=20)
X = pca.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.5, random_state=42)


# Initialize a classifier
# To use TabPFN v2:
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)


# Predict probabilities
prediction_probabilities = clf.predict_proba(X_test)
# For multiclass, pass the full probability matrix
# print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities, multi_class='ovo'))

# Predict labels
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))

/home/nezvevik/projects/TabPFN/src/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


Accuracy 0.902


In [56]:
from sklearn.datasets import fetch_openml
mnist = fetch_openml('mnist_784', version=1)
num_data = 1000
mnist.data = mnist.data[:num_data]
mnist.target = mnist.target[:num_data]
X, y = mnist.data, mnist.target.astype(int)
print(X.shape, y.shape)

# normalize X
X = X / 255.0
X = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

# from sklearn.decomposition import PCA
# pca = PCA(n_components=20)

# X = pca.fit_transform(X)

(1000, 784) (1000,)


In [59]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.6, random_state=42)
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

(400, 784) (600, 784) (400,) (600,)


## TabPFN

In [ ]:
clf = TabPFNClassifier.create_default_for_version(ModelVersion.V2)
clf.fit(X_train, y_train)

# Predict labels
prediction_probabilities = clf.predict_proba(X_test)
predictions = clf.predict(X_test)
print("Accuracy", accuracy_score(y_test, predictions))
print("ROC AUC:", roc_auc_score(y_test, prediction_probabilities, multi_class='ovr'))

/home/nezvevik/projects/TabPFN/src/tabpfn/validation.py:56: UserWarning: Running on CPU with more than 200 samples may be slow.
Consider using a GPU or the tabpfn-client API: https://github.com/PriorLabs/tabpfn-client
  _validate_num_samples_for_cpu(


KeyboardInterrupt: 

## Neural Network

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
class Classifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.3),

            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        return self.net(x)



# Convert DataFrames/Series to tensors
if isinstance(X_train, pd.DataFrame):
    X_train_torch = torch.from_numpy(X_train.to_numpy()).float()
    X_val_torch = torch.from_numpy(X_test.to_numpy()).float()
else:
    # If already NumPy array
    X_train_torch = torch.from_numpy(X_train).float()
    X_val_torch = torch.from_numpy(X_test).float()

if isinstance(y_train, pd.Series):
    y_train_torch = torch.from_numpy(y_train.to_numpy()).long()
    y_val_torch = torch.from_numpy(y_test.to_numpy()).long()
else:
    y_train_torch = torch.from_numpy(np.array(y_train)).long()
    y_val_torch = torch.from_numpy(np.array(y_test)).long()
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

model = Classifier(input_dim=X_train.shape[1], num_classes=10).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)

epochs = 1000
batch_size = 16

for epoch in range(epochs):
    model.train()
    perm = torch.randperm(X_train_torch.size(0))

    total_loss = 0.0

    for i in range(0, X_train_torch.size(0), batch_size):
        idx = perm[i:i + batch_size]

        xb = X_train_torch[idx].to(device)
        yb = y_train_torch[idx].to(device)

        optimizer.zero_grad()
        logits = model(xb)
        loss = criterion(logits, yb)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    # Validation
    model.eval()
    with torch.no_grad():
        logits = model(X_val_torch.to(device))
        preds = torch.argmax(logits, dim=1)
        acc = (preds.cpu() == y_val_torch).float().mean().item()

    print(
        f"Epoch {epoch+1:03d} | "
        f"Loss: {total_loss:.4f} | "
        f"Val Acc: {acc:.3f}"
    )

model.eval()
with torch.no_grad():
    logits = model(X_val_torch.to(device))
    # Convert logits to probabilities using softmax
    probabilities = torch.softmax(logits, dim=1).cpu().numpy()
    nn_predictions = torch.argmax(logits, dim=1).cpu().numpy()

# Compute metrics
nn_accuracy = accuracy_score(y_test, nn_predictions)
nn_roc_auc = roc_auc_score(y_test, probabilities, multi_class='ovr')

print(f"Neural Network Accuracy: {nn_accuracy:.4f}")
print(f"Neural Network ROC AUC: {nn_roc_auc:.4f}")

Epoch 001 | Loss: 45.2184 | Val Acc: 0.663
Epoch 002 | Loss: 25.0784 | Val Acc: 0.767
Epoch 003 | Loss: 14.1445 | Val Acc: 0.802
Epoch 004 | Loss: 7.2826 | Val Acc: 0.820
Epoch 005 | Loss: 4.6160 | Val Acc: 0.820
Epoch 006 | Loss: 2.9820 | Val Acc: 0.823
Epoch 007 | Loss: 2.3545 | Val Acc: 0.810
Epoch 008 | Loss: 1.2863 | Val Acc: 0.815
Epoch 009 | Loss: 0.8485 | Val Acc: 0.822
Epoch 010 | Loss: 0.6847 | Val Acc: 0.815
Epoch 011 | Loss: 0.4884 | Val Acc: 0.803
Epoch 012 | Loss: 0.6630 | Val Acc: 0.798
Epoch 013 | Loss: 0.6730 | Val Acc: 0.805
Epoch 014 | Loss: 0.4748 | Val Acc: 0.800
Epoch 015 | Loss: 0.3859 | Val Acc: 0.807
Epoch 016 | Loss: 0.4566 | Val Acc: 0.802
Epoch 017 | Loss: 0.2362 | Val Acc: 0.798
Epoch 018 | Loss: 0.2628 | Val Acc: 0.810
Epoch 019 | Loss: 0.2634 | Val Acc: 0.807
Epoch 020 | Loss: 0.2091 | Val Acc: 0.807
Epoch 021 | Loss: 0.2440 | Val Acc: 0.807
Epoch 022 | Loss: 0.1870 | Val Acc: 0.807
Epoch 023 | Loss: 0.4091 | Val Acc: 0.805
Epoch 024 | Loss: 0.2392 | Val 

In [54]:
# Random Forest Classifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import time

# Initialize Random Forest
rf_clf = RandomForestClassifier(
    n_estimators=100,
    max_depth=20,
    random_state=42,
    n_jobs=-1,  # Use all available cores
    verbose=1
)

# Train the model
print("Training Random Forest Classifier...")
start_time = time.time()
rf_clf.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Training completed in {train_time:.2f} seconds\n")

# Predict labels
rf_predictions = rf_clf.predict(X_test)
rf_accuracy = accuracy_score(y_test, rf_predictions)
print(f"Accuracy: {rf_accuracy:.4f}")

# Predict probabilities and compute ROC AUC
rf_prediction_probabilities = rf_clf.predict_proba(X_test)
rf_roc_auc = roc_auc_score(y_test, rf_prediction_probabilities, multi_class='ovr')
print(f"ROC AUC: {rf_roc_auc:.4f}")

Training Random Forest Classifier...


[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=-1)]: Done  34 tasks      | elapsed:    0.1s
[Parallel(n_jobs=-1)]: Done 100 out of 100 | elapsed:    0.2s finished
[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


Training completed in 0.23 seconds

Accuracy: 0.8150
ROC AUC: 0.9775


[Parallel(n_jobs=8)]: Using backend ThreadingBackend with 8 concurrent workers.
[Parallel(n_jobs=8)]: Done  34 tasks      | elapsed:    0.0s
[Parallel(n_jobs=8)]: Done 100 out of 100 | elapsed:    0.0s finished


In [55]:
# Gradient Boosting Classifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import accuracy_score, roc_auc_score
import time

# Initialize Gradient Boosting
gb_clf = GradientBoostingClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=5,
    random_state=42,
    verbose=1
)

# Train the model
print("Training Gradient Boosting Classifier...")
start_time = time.time()
gb_clf.fit(X_train, y_train)
train_time = time.time() - start_time
print(f"Training completed in {train_time:.2f} seconds\n")

# Predict labels
gb_predictions = gb_clf.predict(X_test)
gb_accuracy = accuracy_score(y_test, gb_predictions)
print(f"Accuracy: {gb_accuracy:.4f}")

# Predict probabilities and compute ROC AUC
gb_prediction_probabilities = gb_clf.predict_proba(X_test)
gb_roc_auc = roc_auc_score(y_test, gb_prediction_probabilities, multi_class='ovr')
print(f"ROC AUC: {gb_roc_auc:.4f}")

Training Gradient Boosting Classifier...
      Iter       Train Loss   Remaining Time 
         1           1.5453            5.21s
         2           1.2333            4.92s
         3           1.0141            4.81s
         4           0.8381            4.69s
         5           0.7004            4.72s
         6           0.5876            4.68s
         7           0.4958            4.60s
         8           0.4188            4.54s
         9           0.3554            4.54s
        10           0.3003            4.54s
        20           0.0586            4.11s
        30           0.0125            3.60s
        40           0.0029            3.14s
        50           0.0007            2.57s
        60           0.0002            2.03s
        70           0.0000            1.51s
        80           0.0000            1.00s
        90           0.0000            0.50s
       100           0.0000            0.00s
Training completed in 4.94 seconds

Accuracy: 0.7517
ROC A